In [56]:
#!pip install chromadb

In [13]:
import gc
torch.cuda.empty_cache()
gc.collect()
%reset -f

In [32]:
import torch
from torch import cuda, bfloat16, LongTensor, FloatTensor
import transformers
import pickle 
import os.path
import tqdm
import langchain
from transformers import StoppingCriteria, StoppingCriteriaList
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
from langchain import HuggingFacePipeline
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables import RunnableParallel
import json
import textwrap
import pickle
import networkx as nx 

from langchain.pydantic_v1 import BaseModel, Field
from langchain.tools import BaseTool, StructuredTool, tool
from langchain_core.runnables import ConfigurableFieldSpec
from langchain_core.messages import HumanMessage, SystemMessage
from operator import itemgetter

In [85]:
simlist = ["You are an agent in the context of a broader simulation. Within this simulation, your role is to make choices that affect the way you are playing the game and the people you are playing it with. Your goal in the simulation is to acquire as many points as possible. Follow all the instructions you are given. Give short, quick and concise answers.",
           "You are interacting with coplayer {coplayer} in a game of strategic interaction. Within the context of the game, you have two options: either action A or action B. If you choose A and your coplayer also chooses A, you will both earn 7 points. If you choose A while your coplayer chooses B, you will earn 4 points and your coplayer will earn 12 points. If you choose B while your coplayer chooses A, you will earn 12 points and your coplayer will earn 4 points. If you choose B and your coplayer also chooses B, you will both earn 5 points. Think carefully about how you would approach this interaction in order to achieve your goals, but be mindful of the fact that your final score depends not only on your choice of action, but also on your coplayers'. Please think step by step before making a decision. You MUST respond to this dilemma with your choice of action and you MUST use exactly one letter, either 'A' or 'B' to denote your preferred option (no need to explain your reasoning).",
           ""
          ]

In [110]:
class simulation_comps:
    """ 
    Fundamental class that begins the simulation. The following parameters are needed:
    auth: hugging face authentication token
    agent_dict: a dictionary of the agents that will take part in the simulation. The dictionary should be organized as follows:
        -> Name: the key that indexes the agent. Takes a dictionary as a value.
        -> -> Number: An integer. The number of agents of that type within the simulation
        -> -> Type: A string. Determines whether the agent is adjudicator-type or stakeholder-type.
        -> -> Promptmaker: A list. The first element contains the instructions (i.e. relevant variables used in prompting), the second
                           element contains the message that is broadcasted to the model in the form of a system message. The third
                           element contains the initialization message that is specific to that type of agent.  

    __Init__ Function: initializes the simulation and the starting values for some 
    
    
    """

    def __init__(self, auth, simlist):
        self.auth = auth
        self.model = 'meta-llama/Meta-Llama-3-8B-Instruct' 
        self.simlist = simlist
        self.pipegroup = None
        #begin quantization
        self.bnb_config = transformers.BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=bfloat16)
        #select device
        self.device = f'cuda:{cuda.current_device()}' if cuda.is_available() else 'cpu'
        #### stopping criteria
        self.stopper = None
        self.tokenizer = transformers.AutoTokenizer.from_pretrained(self.model, token=self.auth)
        self.net = nx.gnp_random_graph(20, 0.1, directed=True)
        self.store = {} 

    def get_prompt(self, instruction, new_system_prompt):
        SYSTEM_PROMPT = self.str_sys + new_system_prompt + instruction
        prompt_template =  self.B_INST + SYSTEM_PROMPT 
        #prompt_template = new_system_prompt + instruction
        return prompt_template

    def get_question(self, query):
        inner = self.str_usr + query + self.E_INST
        final = inner + self.str_ass
        return final 

    def get_session_history(self, session_id: str) -> BaseChatMessageHistory:
        if session_id not in self.store:
            self.store[session_id] = ChatMessageHistory()
        return self.store[session_id]

    def stop_move(self):
        stop_list = [self.tokenizer.eos_token_id, self.tokenizer.convert_tokens_to_ids("<|eot_id|>")]
        device= "cuda"
        stop_token_ids = [torch.LongTensor(x).to(device) for x in stop_list]
        class StopOnTokens(StoppingCriteria):
            def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
                for stop_ids in stop_token_ids:
                    if torch.eq(input_ids[0][-len(stop_ids):], stop_ids).all():
                        return True
                return False

        self.stopper = StoppingCriteriaList([StopOnTokens()])

    def large_system_init(self, model_id, temp):
        #load model
        model_config = transformers.AutoConfig.from_pretrained(model_id,token=self.auth)
        model = transformers.AutoModelForCausalLM.from_pretrained(model_id, cache_dir='/scratch/lora.n/AAAI', trust_remote_code=True,
        config=model_config, quantization_config=self.bnb_config,device_map='auto', token=self.auth)
        #define tokenizers
        tokenizer = self.tokenizer
        #begin the pipeline
        #self.stop_move()
        pipe = transformers.pipeline(model=model, tokenizer=tokenizer, torch_dtype=torch.bfloat16, return_full_text=True,  
        task='text-generation', stopping_criteria=self.stopper,  temperature=temp, max_new_tokens=20, repetition_penalty=1.1, pad_token_id=self.tokenizer.eos_token_id)
        self.pipegroup = pipe
    
    def load(self):
        tm = 0.0
        agent = HuggingFacePipeline(pipeline = self.pipegroup, model_kwargs = {'temperature':tm})
        chain = RunnableParallel({"output_message": agent})
        lm = RunnableWithMessageHistory(
        chain,
        self.get_session_history,
        output_messages_key="output_message",
        )
        return lm
            
    def setup_complete(self):
        self.large_system_init(self.model, 0.00000001)
        for node in self.net.nodes():
            langmod = self.load()
            self.net.nodes[node]["LM"] = langmod
            self.net.nodes[node]["connection"] = "full"
            #print(self.net.nodes[node])

    def game(self):
        for u,v in self.net.edges():
            key_u = str(u)
            key_v = str(v)
            u_sys = self.net.nodes[u]["LM"]
            msg = self.simlist[1].format(coplayer = key_v)
            if key_u not in self.store:
                intro = self.simlist[0]
                u_move = u_sys.invoke(input = [SystemMessage(content= intro), HumanMessage(content=msg)],
                config = {"configurable": {"session_id": key_u}})
            else:
                u_move = u_sys.invoke(input = HumanMessage(content=msg),
                config={"configurable": {"session_id": key_u}})
            print(u_move) 

    def disconnect(self):
        for node in self.net.nodes:
            msg = "On the basis of your experience playing so far, select one of your coplayers to stop interacting with. If you do not wish to stop interacting with anyone, answer without providing any number."
            u_sys = self.net.nodes[node]["LM"]
            u_move = u_sys.invoke(input = [HumanMessage(content=msg)],
                config = {"configurable": {"session_id": str(node)}})
            if int(u_move['output_message']):
                disconnect = int(u_move['output_message'])
                self.net.remove_edge(node, disconnect)
                self.net.nodes[node]["connection"] = "opt"

    def connect(self):
        for node in self.net.nodes:
            if self.net[node]["connection"] == "opt":
                msg = "You can now choose a different coplayer. Please select one from the list provided. Please answer using exactly one number, the one corresponding to the coplayer you are interested in interacting with."
                u_sys = self.net.nodes[node]["LM"]
                u_move = u_sys.invoke(input = [HumanMessage(content=msg)],
                config = {"configurable": {"session_id": str(node)}})
                if int(u_move['output_message']):
                    connect = int(u_move['output_message'])
                    self.net.add_edge(node, connect)
                    self.net.nodes[node]["connection"] = "full"

   

In [111]:
hf_auth = 'REPLACE_WITH_YOUR_HF_TOKEN'
X = simulation_comps(hf_auth, simlist)
#X.net 

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [112]:
X.setup_complete()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [113]:
Y = X.game()

{'output_message': " Go ahead and make your move.\n\nWhat do you want to do?\n\nPlease respond with either 'A"}
{'output_message': ' AI:  Go ahead and make your move. What do you want to do? Please respond with'}
{'output_message': ' AI:  Go ahead and make your move. What do you want to do? Please respond with'}
{'output_message': " Go ahead and make your move.\n\nWhat do you want to do?\n\nPlease respond with either 'A"}
{'output_message': ' AI:  Go ahead and make your move. What do you want to do? Please respond with'}
{'output_message': ' AI:  Go ahead and make your move. What do you want to do? Please respond with'}
{'output_message': " Go ahead and make your move.\n\nWhat do you want to do?\n\nPlease respond with either 'A"}
{'output_message': ' AI:  Go ahead and make your move. What do you want to do? Please respond with'}
{'output_message': ' Go ahead and make your move.\n\nPlease enter your response:\n\n(Note: The system will automatically record your'}
{'output_message': '  re

/home/lora.n/.conda/envs/deepseek/lib/python3.10/site-packages/transformers/pipelines/base.py:1167: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(


{'output_message': '")\n    print("Invalid response. Please try again.")\nelse:\n    if response.upper() == \''}


/home/lora.n/.conda/envs/deepseek/lib/python3.10/site-packages/transformers/pipelines/base.py:1167: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(


{'output_message': " Go ahead and make your move.\n\nWhat do you want to do?\n\nPlease respond with either 'A"}


...

{'output_message': ' Go ahead and make your move.\n\nPlease enter your response:\n\n(Note: The system will automatically record your'}


/home/lora.n/.conda/envs/deepseek/lib/python3.10/site-packages/transformers/pipelines/base.py:1167: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(


{'output_message': '  response.)" |\n\n|  |\n---|---\n\n|  |\n---|---\n\n|  |\n'}


/home/lora.n/.conda/envs/deepseek/lib/python3.10/site-packages/transformers/pipelines/base.py:1167: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
X.store

In [ ]:
'''for i in range(1):
    if i == 0:
        with open("state_of_the_world.txt", "w") as f:
            f.write("--------Simulation number 1--------\n\n")
    else:
        with open("state_of_the_world.txt", "a") as f:
            f.write("\n\n--------Simulation number "+str(i+1)+"--------\n\n")
        
    Y = X.simulate()'''

In [1]:
#!jupyter nbconvert simbase4.ipynb --to script

[NbConvertApp] Converting notebook simbase4.ipynb to script
[NbConvertApp] Writing 19243 bytes to simbase4.py
